# MongoDB Sharded Cluster

Un espacio para **jugar** con el clúster fragmentado: ver a qué shard llega cada consulta, escribir documentos y
ver en qué shard quedan, leer la tabla de chunks, probar shard keys distintas en tu propia colección y mover rangos
entre shards. Las lecciones (`lessons/`) muestran cada tema en orden; aquí puedes cambiar cualquier cosa y volver a
ejecutar.

**Cómo usarlo**

1. Ejecuta las secciones 0 y 1 (configuración y herramientas) una vez.
2. Después salta a la sección que quieras. Las celdas marcadas con **Juega aquí** están hechas para que las cambies.
3. Si algo queda raro, `python lessons/00_seed.py` deja la base `shop` como al principio. La sección 10 limpia lo
   que crea este notebook.

> Requisito: haber ejecutado `python lessons/00_seed.py` al menos una vez.

| # | Sección | Qué puedes probar |
| --- | --- | --- |
| 0 | Configuración | Conexión a `mongos` y shards registrados |
| 1 | Herramientas | `ruta`, `reparto`, `mapa_chunks`, `en_cada_shard`, `tabla` |
| 2 | Explorar la colección | `find`, `aggregate`, proyecciones |
| 3 | Enrutamiento | Tus filtros: `SINGLE_SHARD` o `SHARD_MERGE` |
| 4 | CRUD a través de mongos | Escribir órdenes y ver en qué shard quedan; cambiar la shard key |
| 5 | Agregaciones | Tu pipeline y cómo lo divide mongos |
| 6 | Metadatos | Tabla de chunks, países por shard, gráfico del reparto |
| 7 | Tu propia shard key | Hashed vs por rango vs clave creciente, con tus datos |
| 8 | Balancer y migraciones | Mover un país de shard, detener y reanudar el balancer |
| 9 | Un shard caído | Qué consultas siguen funcionando si apagas un shard |
| 10 | Limpieza | Borrar lo creado aquí |

## 0. Configuración

La aplicación solo conoce `mongos` (`MONGO_URI` = `mongodb://mongos:27017`). Para mirar qué guarda cada shard, las
herramientas abren además conexiones **directas** a `shard1` y `shard2`, solo para leer. El notebook usa las mismas
funciones que las lecciones (`src/lab/`).

In [ ]:
import random
import time
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import pandas as pd
from bson import MinKey
from pymongo import MongoClient
from pymongo.errors import OperationFailure, PyMongoError

from lab.client import get_client
from lab.cluster import chunks, distribution, route
from lab.config import MONGO_DB, MONGO_URI

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

client = get_client()
db = client[MONGO_DB]
orders = db.orders

print("URI (mongos):", MONGO_URI)
print("MongoDB:     ", client.server_info()["version"])
print("Base:        ", MONGO_DB, "| colecciones:", db.list_collection_names())
print("Órdenes:     ", orders.count_documents({}))   # estimated_document_count() también contaría los huérfanos
pd.DataFrame([{"shard": s["_id"], "host": s["host"]} for s in client.admin.command("listShards")["shards"]])

## 1. Herramientas

Funciones cortas, a la vista, para que las leas y las cambies.

| Función | Qué hace |
| --- | --- |
| `tabla(cursor_o_lista)` | Convierte resultados de MongoDB en un DataFrame |
| `ruta(filtro)` | Cuántos documentos cumplen el filtro y a qué shards preguntó mongos (`explain`) |
| `reparto(coleccion)` | Documentos y MB que tiene cada shard (`$shardedDataDistribution`) |
| `mapa_chunks(coleccion)` | La tabla de rangos de la shard key y su shard dueño |
| `en_cada_shard(filtro)` | Cuenta los documentos que cumplen el filtro **directamente** en cada shard |
| `esperar_balancer(coleccion)` | Muestra el reparto cada 5 s hasta que se equilibra |

In [ ]:
SHARDS = {"shard1rs": "shard1:27018", "shard2rs": "shard2:27018"}
_conexiones = {}


def tabla(rows):
    return pd.DataFrame(list(rows))


def ruta(filtro, coleccion="orders"):
    return {"resultados": db[coleccion].count_documents(filtro), **route(db, filtro, coleccion)}


def reparto(coleccion="orders"):
    return pd.DataFrame(distribution(client, coleccion))


def mapa_chunks(coleccion="orders"):
    return pd.DataFrame([{"shard": c["shard"], "desde": str(c["min"]), "hasta": str(c["max"])}
                         for c in chunks(client, coleccion)])


def shard(nombre):
    """Conexión directa a un shard. Solo para MIRAR: las escrituras siempre van por mongos."""
    if nombre not in _conexiones:
        _conexiones[nombre] = MongoClient(f"mongodb://{SHARDS[nombre]}/?directConnection=true", tz_aware=True,
                                          serverSelectionTimeoutMS=2000)
    return _conexiones[nombre]


def en_cada_shard(filtro, coleccion="orders"):
    # Justo después de una migración, el shard de origen conserva por ~60 s una copia de lo que movió
    # (documentos huérfanos): mongos los ignora, pero una conexión directa los cuenta.
    filas = []
    for nombre in SHARDS:
        try:
            filas.append({"shard": nombre, "documentos": shard(nombre)[MONGO_DB][coleccion].count_documents(filtro)})
        except PyMongoError as error:
            filas.append({"shard": nombre, "documentos": type(error).__name__})
    return pd.DataFrame(filas)


def esperar_balancer(coleccion="orders", maximo=120, tolerancia_mb=3):
    for _ in range(maximo // 5):
        filas = distribution(client, coleccion)
        print("  " + " | ".join(f"{r['shard']}: {r['documents']:>6} docs {r['MB']:.1f} MB" for r in filas))
        if len(filas) == len(SHARDS) and max(r["MB"] for r in filas) - min(r["MB"] for r in filas) < tolerancia_mb:
            break
        time.sleep(5)


reparto()

## 2. Explorar la colección

In [ ]:
orders.find_one({"country": "Colombia"}, {"_id": 0})

### Juega aquí: cualquier `find`

In [ ]:
filtro = {"country": "Perú", "status": "delivered", "total": {"$gt": 1500}}   # cambia el filtro
proyeccion = {"_id": 0, "order_number": 1, "customer_id": 1, "city": 1, "total": 1, "created_at": 1}
print(orders.count_documents(filtro), "órdenes")
tabla(orders.find(filtro, proyeccion).sort("total", -1).limit(10))

# Ideas:
#   {"items": {"$elemMatch": {"category": "Hogar", "quantity": 3}}}
#   {"$expr": {"$gt": [{"$size": "$items"}, 2]}}
#   {"created_at": {"$gte": datetime(2026, 9, 1, tzinfo=timezone.utc)}}

## 3. Enrutamiento

Con shard key `{country: 1, customer_id: 1}`, un filtro que incluye `country` (el prefijo) va a un solo shard
(`SINGLE_SHARD`). Si no lo incluye, mongos pregunta a todos (`SHARD_MERGE`, *scatter-gather*).

### Juega aquí: tus filtros

In [ ]:
cliente = orders.find_one({"country": "Chile"}, {"customer_id": 1})["customer_id"]
filtros = {                                                    # agrega los tuyos
    "un cliente (shard key completa)": {"country": "Chile", "customer_id": cliente},
    "un país (prefijo)": {"country": "Perú"},
    "dos países ($in)": {"country": {"$in": ["Argentina", "Brasil"]}},
    "rango de países": {"country": {"$gte": "A", "$lt": "C"}},
    "solo customer_id (sin prefijo)": {"customer_id": cliente},
    "estado": {"status": "cancelled"},
    "país + estado": {"country": "México", "status": "paid"},
    "$or entre dos países": {"$or": [{"country": "España"}, {"country": "Ecuador"}]},
}
pd.DataFrame([{"filtro": nombre, **ruta(f)} for nombre, f in filtros.items()])

## 4. CRUD a través de mongos

El código es el de siempre; la diferencia es dónde termina cada documento.

### Juega aquí: tus órdenes

In [ ]:
orders.delete_many({"order_number": {"$gte": 950000}})
mis_ordenes = [                                                    # cambia países y clientes
    {"order_number": 950001, "customer_id": "C90001", "country": "Argentina", "city": "Rosario", "total": 10.0},
    {"order_number": 950002, "customer_id": "C90002", "country": "Colombia", "city": "Cali", "total": 20.0},
    {"order_number": 950003, "customer_id": "C90003", "country": "Perú", "city": "Lima", "total": 30.0},
]
for orden in mis_ordenes:
    orden |= {"status": "pending", "items": [], "created_at": datetime.now(timezone.utc)}
orders.insert_many(mis_ordenes)

pd.DataFrame([{"orden": o["order_number"], "país": o["country"],
               **{s: shard(s)[MONGO_DB].orders.count_documents({"order_number": o["order_number"]}) for s in SHARDS}}
              for o in mis_ordenes])

In [ ]:
# Cambiar el valor de la shard key mueve el documento a otro rango (y quizá a otro shard).
# Debe hacerse con la shard key completa en el filtro.
orders.update_one({"country": "Argentina", "customer_id": "C90001", "order_number": 950001},
                  {"$set": {"country": "Perú"}})                    # prueba otros países
print("ahora la orden 950001 vive en:")
en_cada_shard({"order_number": 950001})

In [ ]:
# update_many sin shard key: mongos lo envía a todos los shards
print("modificadas:", orders.update_many({"order_number": {"$gte": 950000}}, {"$set": {"status": "paid"}}).modified_count)
print(ruta({"order_number": {"$gte": 950000}}))

## 5. Agregaciones

mongos divide el pipeline: cada shard hace lo que puede con sus datos (`shardsPart`) y luego se fusionan los
resultados parciales (`mergerPart`).

### Juega aquí: tu pipeline y su plan

In [ ]:
pipeline = [
    {"$match": {"status": {"$ne": "cancelled"}}},
    {"$unwind": "$items"},
    {"$group": {"_id": {"país": "$country", "categoría": "$items.category"}, "unidades": {"$sum": "$items.quantity"}}},
    {"$sort": {"unidades": -1}},
    {"$limit": 10},
]
display(tabla(orders.aggregate(pipeline)))

explain = db.command("explain", {"aggregate": "orders", "cursor": {}, "pipeline": pipeline}, verbosity="queryPlanner")
if "splitPipeline" in explain and explain["splitPipeline"]:
    print("shards consultados:", sorted(explain["shards"]))
    print("en cada shard:     ", [list(e)[0] for e in explain["splitPipeline"]["shardsPart"]])
    print("al fusionar:       ", [list(e)[0] for e in explain["splitPipeline"]["mergerPart"]])
else:
    print("El pipeline se ejecutó completo en un shard:", sorted(explain.get("shards", {})))

# Prueba a empezar con {"$match": {"country": "Colombia"}}: el pipeline completo va a un solo shard

## 6. Metadatos del clúster

Lo que guarda el config server (base `config`) y usa mongos para enrutar.

In [ ]:
mapa_chunks()

In [ ]:
# Qué países viven en cada shard (según el enrutamiento de mongos)
paises = pd.DataFrame([{"país": p, "shards": route(db, {"country": p})["shards"],
                        "órdenes": orders.count_documents({"country": p})} for p in orders.distinct("country")])
paises

In [ ]:
datos = reparto()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.bar(datos["shard"], datos["documents"])
ax1.set_title("documentos por shard")
ax2.bar(datos["shard"], datos["MB"], color="orange")
ax2.set_title("MB por shard")
plt.tight_layout()

## 7. Tu propia shard key

La shard key es la decisión más importante del clúster. Aquí creas una colección `sandbox_events` con la shard key
que quieras, insertas eventos y ves cómo quedan repartidos y cómo se enrutan las consultas.

Cada evento tiene `user_id` (2.000 usuarios), `type` y `created_at` (siempre creciente, como un timestamp real).

In [ ]:
def probar_shard_key(clave, n=20_000):
    coleccion = db.sandbox_events
    coleccion.drop()
    client.admin.command("shardCollection", f"{MONGO_DB}.sandbox_events", key=clave)
    rng, inicio = random.Random(7), datetime(2026, 1, 1, tzinfo=timezone.utc)
    coleccion.insert_many([{"user_id": f"U{rng.randint(1, 2000):05d}", "type": rng.choice(["click", "view", "buy"]),
                            "created_at": inicio + timedelta(seconds=i), "payload": "x" * 100} for i in range(n)],
                          ordered=False)
    print(f"shard key {clave}: {len(chunks(client, 'sandbox_events'))} chunks")
    display(reparto("sandbox_events"))
    consultas = {
        "un usuario": {"user_id": "U00042"},
        "un rango de fechas": {"created_at": {"$gte": inicio + timedelta(hours=1), "$lt": inicio + timedelta(hours=2)}},
        "un tipo": {"type": "buy"},
    }
    display(pd.DataFrame([{"consulta": k, **ruta(v, "sandbox_events")} for k, v in consultas.items()]))

### Juega aquí: elige la shard key

In [ ]:
probar_shard_key({"user_id": "hashed"})      # reparto uniforme desde el inicio; los rangos de fecha van a todos

# Prueba también:
#   probar_shard_key({"created_at": 1})       clave creciente: todo cae en el último chunk -> un solo shard (hot shard)
#   probar_shard_key({"user_id": 1})          por rango: empieza en un shard y el balancer lo reparte después
#   probar_shard_key({"user_id": 1, "created_at": 1})

In [ ]:
# Después de insertar, el balancer sigue trabajando: observa cómo cambia el reparto
esperar_balancer("sandbox_events", maximo=60)

## 8. Balancer y migraciones

In [ ]:
estado = client.admin.command("balancerStatus")
{"modo": estado["mode"], "en_ronda": estado["inBalancerRound"],
 "chunksize_MB": client.config.settings.find_one({"_id": "chunksize"})["value"]}

### Juega aquí: mover un país a otro shard

In [ ]:
def mover_pais(pais, destino):
    ns = f"{MONGO_DB}.orders"
    siguiente = min((p for p in orders.distinct("country") if p > pais), default=None)
    puntos = [{"country": pais, "customer_id": MinKey()}] + ([{"country": siguiente, "customer_id": MinKey()}] if siguiente else [])
    for punto in puntos:
        try:
            client.admin.command("split", ns, middle=punto)       # cortar justo en los bordes del país
        except OperationFailure:
            pass                                                  # ya era un borde de chunk
    for c in chunks(client):
        if c["min"].get("country") == pais and c["shard"] != destino:
            client.admin.command("moveRange", ns, min=c["min"], max=c["max"], toShard=destino)


pais, destino = "Colombia", "shard2rs"                 # cambia el país y el destino
print("antes: ", route(db, {"country": pais}))
mover_pais(pais, destino)
print("después:", route(db, {"country": pais}))
reparto()

In [ ]:
# Detener y reanudar el balancer (por ejemplo, durante una ventana de mantenimiento)
client.admin.command("balancerStop")
print("balancer:", client.admin.command("balancerStatus")["mode"])
client.admin.command("balancerStart")
print("balancer:", client.admin.command("balancerStatus")["mode"])

## 9. Un shard caído

### Juega aquí: apaga un shard desde el host

En una terminal de tu equipo ejecuta `docker stop MongoShard2`, luego esta celda, y al final
`docker start MongoShard2`. Las consultas dirigidas al shard que sigue vivo funcionan; las que necesitan el shard
apagado fallan.

In [ ]:
for nombre, filtro in {"estado (todos los shards)": {"status": "paid"},
                       **{f"país {p}": {"country": p} for p in ("Perú", "Colombia", "España")}}.items():
    try:
        conteo = orders.count_documents(filtro, maxTimeMS=5000)
        print(f"{nombre:28} -> {conteo} ({route(db, filtro)['shards']})")
    except PyMongoError as error:
        print(f"{nombre:28} -> {type(error).__name__}: {str(error)[:70]}")

## 10. Limpieza

In [ ]:
print("órdenes de prueba:", orders.delete_many({"order_number": {"$gte": 950000}}).deleted_count)
db.drop_collection("sandbox_events")
reparto()

---

**¿Rompiste algo?** `python lessons/00_seed.py` reconstruye la base `shop` (tarda ~15 s esperando al balancer).